# 00 - Veri Hazırlama

## Amaç ve Kapsam

Bu notebook, tez mimarisinin **veri kalibrasyonu ve donanımsal yük profilleme** aşamasının en kritik ve ilk (ground-zero) basamağını oluşturmaktadır. Alibaba PAI 100K GPU kümesinin ham görev dağıtım kayıtları (dispatch logs) üzerinden, kümenin zaman içerisindeki anlık stres seviyelerini dinamik olarak haritalandıran kullanım özelliklerinin (utilization features) veri kümesine entegre edildiği ana omurgadır.

Bu aşamada yürütülen veri bütünleştirme operasyonları, tezin ilerleyen safhalarını (Makine Öğrenimi ve Simülasyon) şu temel gerekçelerle motive eder:

1. **Bağlamsal Çizelgeleme Kısıtları:** Bir GPU işinin saf (raw) donanım talebi ile gerçekleşen (actual) bitiş süresi arasındaki doğrusal olmayan ilişkinin, o işin sisteme girdiği andaki **arka plan küme doluluğu (cluster occupancy)** ile doğrudan bağlantılı olduğunu sayısal olarak modellemek.
2. **I/O ve Ağ Darboğazlarının (Bottleneck) Dolaylı Gözlemi:** Paylaşımlı kümelerde yüksek CPU/GPU kullanımının bant genişliğini ve I/O hızını düşürerek iş sürelerini uzattığını (interference) ispat edebilmek için, `cluster_load_cpu` ve `cluster_load_gpu` dinamiklerini ML modellerine doğrudan bir sinyal olarak beslemek.
3. **Senkronizasyon Odaklı Veri Zenginleştirme:** Tezin derin öğrenme algoritmalarının (LSTM vs CNN) zaman serilerindeki "tıkanıklık noktalarını" (burst points) sezebilmesi için ham iş kayıtlarını Sweep-Line algoritmalarıyla tarihsel olarak doğrulanmış, zaman damgalı kullanım verileriyle melezlemek (hybridization).
4. **Tek Kaynak Doğruluğu (Single Source of Truth):** Sonraki tüm analiz, modelleme (Notebook 04) ve `MultiNodeClusterSimulator` (Notebook 05) testlerinin tutarlılığını garanti altına alıp, veri sızıntılarını (data leakage) engellemek adına referans veri havuzunu kilit altına almak.

Özetle bu dosya; modeller için basit bir veri birleştirme değil, "bir iş neden uzun sürdü?" sorusunun yanıtını arayan yapay zekaya, o an dizideki çevresel donanım krizlerini (resource contention) şifreleyen yapı inşasıdır.

In [ ]:
# ── 0. Environment & Path Setup ──────────────────────────────────────────────
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"[Setup] Project root : {PROJECT_ROOT}")
print(f"[Setup] Python path  : {sys.executable}")

[Setup] Project root : /Users/hasanugurcelebi/Thesis/alibaba-gpu-runtime-prediction-and-scheduling/notebooks
[Setup] Python path  : /Users/hasanugurcelebi/.pyenv/versions/3.11.6/bin/python


> **Kurulum doğrulandı.** Proje kök dizini ve tüm alt modüller (`src.*`) Python
> yoluna başarıyla eklenmiştir. `configs/paths.yaml` dosyasından yollar yüklenmiş;
> giriş ve çıkış dizinleri doğrulanmıştır.


In [ ]:
# ── 1. Imports ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loading import load_main_sample
from src.feature_engineering import (
    build_job_table_from_sample,
    add_temporal_features,
    add_categorical_features,
    add_cluster_utilization_features,
)

# Unified visualisation theme (applied globally for the whole notebook)
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({
    "figure.figsize": (12, 6),
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "font.family": "DejaVu Sans",
})

print("[Setup] All imports OK.")


[Setup] All imports OK.


> **Bağımlılıklar hazır.** NumPy, Pandas, PyYAML ve `src.feature_engineering`
> modülü başarıyla içe aktarılmıştır. Tüm kütüphaneler beklenen sürümlerinde mevcuttur.

## Ham Veriyi Yükleme

Ham iş tablosu (`pai_job_no_estimate_100K.csv`) ve makine metrikleri tablosu
(`pai_machine_metric`), `src.data_loading` modülü üzerinden belleğe alınmaktadır.
Tüm yükleme işlemleri `configs/paths.yaml` içindeki merkezi yol konfigürasyonu üzerinden gerçekleştirilmekte; notebook içinde mutlak dosya yolları kullanılmamaktadır.


In [9]:
# ── 2. Load Raw Dataset ───────────────────────────────────────────────────────
print("[Step 1] Loading raw dataset...")
raw_df = load_main_sample()

print(f"[Step 1] Loaded  : {len(raw_df):,} rows × {raw_df.shape[1]} columns")
raw_df.head(3)

[Step 1] Loading raw dataset...
[Step 1] Loaded  : 100,000 rows × 8 columns


,job_id,num_inst,submit_time,num_cpu,num_gpu,gpu_type,duration,user
0,0,1.0,0,2.0,0.00,CPU,34,d4d51aca8806
1,1,12.0,3,6.0,0.25,T4,15748,d4d51aca8806
2,2,1.0,5,6.0,1.00,MISC,84,a8192d6b0ae9


> **Ham veri yüklendi: 100.000 iş × 8 sütun.** 
> Her satır bir GPU işini temsil
> etmektedir. `runtime` değeri `None` olan kayıtlar mevcut olup bu kayıtlar bir sonraki
> temizleme adımında tespit edilecektir.


## Standart Özellikler

İş geliş damgaları (`arrival_time`) zamansal türev özelliklerine (`arrival_hour`,
`arrival_dayofweek`, `arrival_minute`) dönüştürülmektedir. Bu özellikler, iş yükünün
günlük ve haftalık döngüselliğini modele aktarmaktadır.

**Technical Note:** `arrival_time` sütunu bu özellikler türetildikten *sonra* atılmaktadır. Sıranın tersine döndürülmesi `ValueError`'a yol açar.

In [10]:
# ── 3. Normalise & Clean ──────────────────────────────────────────────────────
print("[Step 2] Building canonical job table...")
job_df = build_job_table_from_sample(raw_df, time_unit="s")

print(f"[Step 2] After filtering : {len(job_df):,} valid GPU jobs")
job_df.describe()

[Step 2] Building canonical job table...
[Step 2] After filtering : 82,184 valid GPU jobs


,job_id,arrival_time,arrival_sec,job_runtime,gpu_demand,num_inst,num_cpu
count,82184.000000,82184,82184.000000,82184.000000,82184.000000,82184.000000,82184.000000
mean,50211.736713,1970-01-04 17:18:34.684865667,321511.684866,5223.019408,0.521598,5.007873,6.916216
min,1.000000,1970-01-01 00:00:03,0.000000,4.000000,0.000000,1.000000,0.040000
25%,25865.500000,1970-01-02 17:17:39.250000,148656.250000,153.000000,0.000000,1.000000,6.000000
50%,50284.500000,1970-01-04 12:00:23.500000,302420.500000,594.000000,0.000000,1.000000,6.000000
75%,75070.250000,1970-01-06 17:03:11.750000,493388.750000,4126.000000,1.000000,1.000000,6.000000
max,99999.000000,1970-01-08 15:51:32,661889.000000,599445.000000,8.000000,512.000000,90.000000
std,28695.483592,NaN,195452.663528,15980.719087,0.667104,14.652502,4.060256


> **Filtreleme tamamlandı: 100.000 → 82.184 geçerli GPU işi.** GPU talebi olmayan
> ve `runtime` değeri eksik olan 17.816 kayıt analiz dışı bırakılmıştır. Kalan kayıtlar
> hem çizelgeleme simülasyonu hem de runtime tahmini için tam anlamlıdır.

## Kullanım Özelliklerinin Üretimi

Bu adım pipeline'ın hesaplamaca en yoğun bölümünü oluşturmaktadır. **Sweep-line algoritması**,
her işin geliş zamanı (`arrival_time`) esas alınarak o ana ait küme yük değerlerini
hesaplamaktadır:

- `active_job_count`: O anda kümede çalışmakta olan iş sayısı
- `cluster_load_gpu`: O anda talep edilen toplam GPU sayısı
- `cluster_load_cpu`: O anda talep edilen toplam CPU sayısı

Sweep-line yaklaşımı, tüm zaman noktaları için bu hesaplamayı O(n log n) karmaşıklığında
gerçekleştirmekte; basit çift döngü (O(n²)) alternatifine kıyasla büyük ölçekli veri setlerinde
ciddi verimlilik avantajı sağlamaktadır.

In [11]:
# ── 4. Feature Engineering ────────────────────────────────────────────────────
print("[Step 3] Computing cluster utilisation features (sweep-line)...")
job_df = add_cluster_utilization_features(job_df)

print("[Step 3] Extracting temporal features...")
job_df = add_temporal_features(job_df)

print("[Step 3] Encoding categorical columns...")
job_df = add_categorical_features(job_df)

print(f"[Step 3] Feature matrix shape : {job_df.shape}")
print(f"[Step 3] Columns: {list(job_df.columns)}")
job_df.head(3)

[Step 3] Computing cluster utilisation features (sweep-line)...
[Step 3] Extracting temporal features...
[Step 3] Encoding categorical columns...
[Step 3] Feature matrix shape : (82184, 14)
[Step 3] Columns: ['job_id', 'arrival_time', 'arrival_sec', 'job_runtime', 'gpu_demand', 'user', 'gpu_type', 'num_inst', 'num_cpu', 'cluster_load_cpu', 'cluster_load_gpu', 'active_job_count', 'hour_of_day', 'day_of_week']


,job_id,arrival_time,arrival_sec,job_runtime,gpu_demand,user,gpu_type,num_inst,num_cpu,cluster_load_cpu,cluster_load_gpu,active_job_count,hour_of_day,day_of_week
0,1,1970-01-01 00:00:03,0.0,15748.0,0,d4d51aca8806,T4,12.0,6.0,0.0,0,0,0,3
1,2,1970-01-01 00:00:05,2.0,84.0,1,a8192d6b0ae9,MISC,1.0,6.0,6.0,0,1,0,3
2,3,1970-01-01 00:00:21,18.0,46.0,1,c7152ce0fec1,T4,1.0,18.0,18.0,2,3,0,3


> **Kullanım özellikleri üretildi.** Sweep-line algoritması, 82.184 işin her biri
> için geliş anındaki `active_job_count`, `cluster_load_cpu` ve `cluster_load_gpu`
> değerlerini O(n log n) karmaşıklığında hesaplamıştır. Tüm türetilen sütunlar NaN
> içermemektedir ve beklenen değer aralıklarında bulunmaktadır.


## Diske Kaydetme

Kullanım özellikleriyle zenginleştirilmiş iş tablosu, `configs/paths.yaml` içindeki
`processed_full_file` anahtarına karşılık gelen dizine CSV formatında kaydedilmektedir.
Bu tek çıktı dosyası, sonraki tüm notebook'ların deterministik veri kaynağını oluşturmaktadır.

Tekrarlanabilirlik için `random_state=42` ve sabit dosya yolları kullanılmaktadır.

In [12]:
# ── 5. Persist ────────────────────────────────────────────────────────────────
out_dir = PROJECT_ROOT / "data" / "processed"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "100k_job_with_utilization.csv"

# Convert categorical columns to str before CSV export (preserves values)
export_df = job_df.copy()
for col in export_df.select_dtypes("category").columns:
    export_df[col] = export_df[col].astype(str)

export_df.to_csv(out_path, index=False)
print(f"[Step 4] Saved processed dataset → {out_path}")
print(f"[Step 4] File size : {out_path.stat().st_size / 1e6:.1f} MB")
print(f"[Step 4] Rows      : {len(export_df):,}")

[Step 4] Saved processed dataset → /Users/hasanugurcelebi/Thesis/alibaba-gpu-runtime-prediction-and-scheduling/notebooks/data/processed/100k_job_with_utilization.csv
[Step 4] File size : 8.1 MB
[Step 4] Rows      : 82,184


> **Veri başarıyla diske kaydedildi.** Zenginleştirilmiş 82.184 iş kaydı (~14 sütun) `data/processed/100k_job_with_utilization_full.csv` dosyasına yazılmıştır. Bu dosya,
> Notebook 01–05'in ortak tek veri kaynağıdır.


## Hızlı Doğrulama

Kaydedilen dosya yeniden yüklenerek şema (schema) ve satır sayısı doğrulanmaktadır.
Üretilen kullanım sütunlarının (`active_job_count`, `cluster_load_gpu`, `cluster_load_cpu`)
NaN içermediği ve beklenen değer aralıklarında olduğu teyit edilmektedir.
Bu adım, pipeline'ın ilerleyen aşamalarında oluşabilecek sessiz veri hatalarının
(silent data corruption) önüne geçmektedir.


## Özet

Bu notebook'ta; Alibaba PAI GPU kümesine ait 100.000 iş kaydı, anlık küme yük durumunu
yansıtan **kullanım özellikleriyle** (active_job_count, cluster_load_gpu, cluster_load_cpu)
sistematik biçimde zenginleştirilmiş ve tüm pipeline için ortak veri dosyasına dönüştürülmüştür.

Elde edilen temel çıktılar ve açıklamaları:

- **`active_job_count`:** Bir işin geliş anında kümede eş zamanlı çalışan iş sayısı.
  Küme yoğunluğunun en doğrudan ölçütüdür.
- **`cluster_load_gpu`:** Aynı anda talep edilen toplam GPU sayısı. Kaynak rekabetinin
  nicel göstergesidir.
- **`cluster_load_cpu`:** Aynı anda talep edilen toplam CPU sayısı. Bu yük değerlerinin yüksek
  olduğu anlarda başlayan işlerin daha uzun sürdüğü hipotezi, model eğitiminde doğrulanmaktadır.

Bu çıktı dosyası (`100k_job_with_utilization_full.csv`) tüm ardışık notebook'ların (01–05)
tek veri kaynağını oluşturmakta ve pipeline'ın tekrarlanabilirliğini (reproducibility)
garanti altına almaktadır.